# check what folder we are on 

In [1]:
import os
os.getcwd()

'/Users/yerik/_apple_lib/_a_progs/_a2ms_env/_9_ML_project'

# PROJECT STRUCTURE INITIALIZER

In [2]:
# ---------------------------------------------------
# PROJECT STRUCTURE INITIALIZER
# ---------------------------------------------------

import os
from pathlib import Path

# Base folders we want to create
folders = [
    "data",
    "src",
    "reports"
]

# Create folders if they do not exist
for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)
    print(f"Created: {folder}")

print("\nStructure initialized.")

Created: data
Created: src
Created: reports

Structure initialized.


# add DATA / REPORTS sub-folders structure 

In [7]:
# ---------------------------------------------------
# DATA STRUCTURE EXTENSION
# ---------------------------------------------------

from pathlib import Path

folders = [
    "data/raw",
    "data/processed",
    "reports/tables"
]

for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)
    print(f"Created: {folder}")

print("\nData structure ready.")

Created: data/raw
Created: data/processed
Created: reports/tables

Data structure ready.


# reports and figures folder 

In [4]:
# ---------------------------------------------------
# REPORTS STRUCTURE
# ---------------------------------------------------

from pathlib import Path

folders = [
    "reports/figures"
]

for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)
    print(f"Created: {folder}")

print("\nReports structure ready.")

Created: reports/figures

Reports structure ready.


# 1 # create a module to load data 

In [2]:
# ---------------------------------------------------
# WRITE PKL CONCAT LOADER
# ---------------------------------------------------

from pathlib import Path

code = '''
# ---------------------------------------------------
# PKL CONCAT DATA LOADER MODULE
# ---------------------------------------------------

import pandas as pd
from pathlib import Path
from tqdm import tqdm


def load_pkl_folder(path):
    """
    Load and concatenate all PKL files from a folder (including subfolders).

    Parameters
    ----------
    path : str
        Root directory containing PKL files

    Returns
    -------
    df : pandas.DataFrame
    """

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Folder not found: {path}")

    # ---------------------------------------------------
    # FIND PKL FILES
    # ---------------------------------------------------

    pkl_files = [p for p in path.rglob("*.pkl") if not p.name.startswith("._")]

    if len(pkl_files) == 0:
        raise ValueError("No PKL files found in directory")

    print(f"\\nFound {len(pkl_files)} PKL files")

    # ---------------------------------------------------
    # LOAD + CONCAT
    # ---------------------------------------------------

    dfs = []

    for pkl in tqdm(pkl_files, desc="Loading PKLs"):
        try:
            df = pd.read_pickle(pkl)
            df["__source_file"] = pkl.name
            dfs.append(df)
        except Exception as e:
            print(f"Error loading {pkl}: {e}")

    if len(dfs) == 0:
        raise ValueError("No valid PKL files could be loaded")

    df = pd.concat(dfs, ignore_index=True)

    # ---------------------------------------------------
    # INFO
    # ---------------------------------------------------

    print("\\nDataset Combined")
    print("---------------------------")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory usage: {mem:.2f} MB")

    return df
'''

Path("src/data_loader_pkl.py").write_text(code)

print("Updated: src/data_loader_pkl.py")

Updated: src/data_loader_pkl.py


# create module to perform EDA pt.1

In [9]:
# ---------------------------------------------------
# CREATE EDA MODULE
# ---------------------------------------------------

from pathlib import Path

code = '''
# ---------------------------------------------------------
# SIMPLE EDA (STEPS 1–6) + REPORT EXPORT
# does NOT modify df
# ---------------------------------------------------------

import pandas as pd
from pathlib import Path

def eda_01_06_GET_report(df, path_out="reports/tables/eda_summary.csv"):

    # ------------------------------------------------
    # Console overview
    # ------------------------------------------------
    print("\\nDATA SHAPE:", df.shape)
    print("\\nCOLUMNS:", list(df.columns))

    print("\\nHEAD:\\n", df.head())
    print("\\nTAIL:\\n", df.tail())

    print("\\nDATA INFO:")
    df.info()

    print("\\nDUPLICATE ROWS:", df.duplicated().sum())

    # ------------------------------------------------
    # Target overview
    # ------------------------------------------------
    if "purchase" in df.columns:
        print("\\nTARGET DISTRIBUTION:")
        print(df["purchase"].value_counts())

    # ------------------------------------------------
    # Report table
    # ------------------------------------------------
    report = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.values,
        "missing": df.isna().sum().values,
        "missing_%": (df.isna().mean()*100).round(2).values,
        "unique": df.nunique().values
    })

    # ------------------------------------------------
    # Save report
    # ------------------------------------------------
    Path(path_out).parent.mkdir(parents=True, exist_ok=True)
    report.to_csv(path_out, index=False)

    print("\\nREPORT SAVED →", path_out)

    return report
'''

Path("src/eda.py").write_text(code)

print("Created: src/eda.py")

Created: src/eda.py


# Preprocess data , x , y variable 

In [10]:
from pathlib import Path

code = '''
# ---------------------------------------------------------
# PREPROCESSING
# encode categoricals + split features/target
# ---------------------------------------------------------

import pandas as pd

def prep_features_target(df, target="purchase"):

    # separate X and y
    X = df.drop(columns=[target])
    y = df[target]

    # encode categorical variables
    X = pd.get_dummies(X, drop_first=True)

    print("\\nFEATURE MATRIX:", X.shape)
    print("TARGET:", y.shape)

    return X, y
'''

Path("src/preprocessing.py").write_text(code)

print("Created: src/preprocessing.py")

Created: src/preprocessing.py


# model Preparation 

In [12]:
from pathlib import Path

code = '''
# ---------------------------------------------------------
# MODEL PREP
# train / test split
# ---------------------------------------------------------

from sklearn.model_selection import train_test_split

def split_train_test(X, y, test_size=0.2, random_state=42):

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state
    )

    print("\\nTRAIN:", X_train.shape)
    print("TEST:", X_test.shape)

    return X_train, X_test, y_train, y_test
'''

Path("src/modeling.py").write_text(code)

print("Created: src/modeling.py")

Created: src/modeling.py
